# Liquidity Stress Prediction: v5 (more diverse strong models, runway features, honest stacking)

Built on v2 (blend CV 0.23055 nested / 0.92030), which together with v1 scored best on the leaderboard.

**Why v3/v4 calibration did not carry over to the leaderboard.** The recalibration map was fitted on out-of-fold predictions,
where each row is scored by 3 fold models (one per seed). Test predictions are the average of 30 fold models (3 seeds x 10 folds)
averaged in probability space, which pulls them toward the middle. The two distributions differ, so a map fitted on one does not
fit the other, and its CV gain (0.0003) is smaller than public-leaderboard noise anyway. v5 drops it.

**What v5 changes.** Log loss and AUC both only move when the blend gets new information, so v5 adds:
1. **Runway / projection features** (balance M1 against the recent monthly deficit, next-month balance if flows repeat,
   largest single outflow vs balance). They go into a separate `plus` view so the v2 feature ranking and caches stay valid.
   A quick single-model check prints whether they help before the long runs.
2. **Three new models** that are as strong as the existing ones but err differently:
   XGBoost (depth-wise trees, was trained in v1 but never blended), LightGBM with extra-trees on the `plus` view,
   and a deeper CatBoost on the `plus` view. New models average fold predictions in **logit space**, which keeps test
   predictions on the same scale as the out-of-fold ones.
3. **Blend chosen honestly** among: constrained weights (v2), greedy forward selection, and a logistic stack
   (free weights + intercept). Each is scored with weights fitted out of sample; the best is kept.
4. Optional **pseudo-labelling** of confident test rows (off by default: CV cannot measure it fairly, check it on the leaderboard).
5. A free extra file: `submission_v1v2_logitavg.csv`, the logit average of `submissionv1.csv` and `submissionv2.csv`
   (different fold splits and fold counts, so averaging them is extra bagging).

The cached v2/v3 predictions in `preds/` (lgb, cb, cb_ord, nn) are reused, so only the new models train.
Expected files in DATA_DIR: `Train.csv`, `Test.csv`. All randomness is seeded.

In [ ]:
import os, warnings, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
from scipy.optimize import minimize
from scipy.special import logit, expit
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.preprocessing import QuantileTransformer

warnings.filterwarnings("ignore")

# ---------------- settings ----------------
DATA_DIR = r"C:\Users\23108\Downloads\mobile money project"
SEED = 50                        # feature-selection seed (same as v1-v3, keeps the cache valid)
N_FOLDS = 10
SEEDS_LGB = [42, 7, 2026]        # cached from v2/v3
SEEDS_CB = [42, 7, 2026]         # cached
SEEDS_CB_ORD = [42, 7]           # cached
SEEDS_NN = [42, 7, 2026]         # cached
SEEDS_XGB = [42, 7, 2026]        # new
SEEDS_LGB_PLUS = [42, 7, 2026]   # new
SEEDS_CB_PLUS = [42, 7]          # new (slowest)
RUN_PSEUDO = False               # pseudo-labelling of confident test rows (see its section)
THREADS = -1
CACHE_DIR = "preds"
# ------------------------------------------

if os.path.isdir(DATA_DIR):
    os.chdir(DATA_DIR)
os.makedirs(CACHE_DIR, exist_ok=True)
np.random.seed(SEED)
print("working dir:", os.getcwd())
print("lightgbm", lgb.__version__, "| xgboost", xgb.__version__)

In [ ]:
TEST_FILE = "Test.csv" if os.path.exists("Test.csv") else "Test (1).csv"
train = pd.read_csv("Train.csv")
test = pd.read_csv(TEST_FILE)
TARGET, ID_COL = "liquidity_stress_next_30d", "ID"
CATS = ["gender", "region", "smartphone", "segment", "earning_pattern"]
y = train[TARGET].values
print(train.shape, test.shape, "positive rate:", y.mean().round(4), "| test file:", TEST_FILE)

## Feature engineering
M1 is the most recent month and M6 the oldest. Inflow channels: P2P received, cash deposit, bank transfer in.
Outflow channels: P2P send, cash withdrawal, paybill, merchant pay.

In [ ]:
CH_IN = ["received", "deposit", "transfer_from_bank"]
CH_OUT = ["mm_send", "withdraw", "paybill", "merchantpay"]
CH = CH_IN + CH_OUT
UNIQ = {"paybill": "companies", "merchantpay": "merchants", "transfer_from_bank": "banks",
        "mm_send": "recipients", "received": "senders", "deposit": "agents", "withdraw": "agents"}
MONTHS = range(1, 7)
TC = np.arange(1, 7)[::-1].astype(float); TC = TC - TC.mean()   # time index, oldest -> newest
E = 1.0  # smoothing constant for ratios


def slope(A):
    """OLS slope over the 6 months (columns ordered m1..m6), positive = rising toward the present."""
    return (A * TC).sum(1) / (TC ** 2).sum()


def mat(df, fmt):
    return np.column_stack([df[fmt.format(m)].values.astype(float) for m in MONTHS])


def build_features(df):
    o = {}
    B = mat(df, "m{}_daily_avg_bal")
    bal_mean, bal_min = B.mean(1), B.min(1)
    o["bal_mean"] = bal_mean; o["bal_std"] = B.std(1); o["bal_min"] = bal_min; o["bal_max"] = B.max(1)
    o["bal_slope"] = slope(B); o["bal_slope_rel"] = o["bal_slope"] / (bal_mean + E)
    o["bal_cv"] = o["bal_std"] / (bal_mean + E)
    o["bal_m1_m6"] = B[:, 0] / (B[:, 5] + E); o["bal_m1_m2"] = B[:, 0] / (B[:, 1] + E)
    o["bal_m1_mean"] = B[:, 0] / (bal_mean + E); o["bal_m13_m46"] = B[:, :3].mean(1) / (B[:, 3:].mean(1) + E)
    o["bal_argmin"] = B.argmin(1); o["bal_argmax"] = B.argmax(1)
    o["bal_ndecl"] = (np.diff(B[:, ::-1], axis=1) < 0).sum(1)
    o["bal_m1_min"] = B[:, 0] / (bal_min + E)
    for k in range(5):
        o[f"bal_d{k+1}"] = B[:, k] - B[:, k + 1]

    V, TV = {}, {}
    for c in CH:
        v = V[c] = mat(df, "m{}_" + c + "_volume")
        tv = TV[c] = mat(df, "m{}_" + c + "_total_value")
        H = mat(df, "m{}_" + c + "_highest_amount")
        U = mat(df, "m{}_" + c + "_" + UNIQ[c])
        o[f"{c}_tv_mean"] = tv.mean(1); o[f"{c}_tv_std"] = tv.std(1); o[f"{c}_tv_slope"] = slope(tv) / (tv.mean(1) + E)
        o[f"{c}_tv_m1_mean"] = tv[:, 0] / (tv.mean(1) + E); o[f"{c}_tv_m13_m46"] = tv[:, :3].sum(1) / (tv[:, 3:].sum(1) + E)
        o[f"{c}_v_mean"] = v.mean(1); o[f"{c}_v_slope"] = slope(v) / (v.mean(1) + E); o[f"{c}_v_m1_mean"] = v[:, 0] / (v.mean(1) + E)
        o[f"{c}_zero_m"] = (v == 0).sum(1); o[f"{c}_last_active"] = np.where((v > 0).any(1), (v > 0).argmax(1), 6)
        o[f"{c}_avg_size"] = tv.sum(1) / (v.sum(1) + E); o[f"{c}_avg_size_m1"] = tv[:, 0] / (v[:, 0] + E)
        o[f"{c}_hi_share"] = H.sum(1) / (tv.sum(1) + E); o[f"{c}_hi_max"] = H.max(1)
        o[f"{c}_uniq_mean"] = U.mean(1); o[f"{c}_uniq_m1_mean"] = U[:, 0] / (U.mean(1) + E)
        o[f"{c}_tv_to_bal"] = tv.mean(1) / (bal_mean + E)

    IN = sum(TV[c] for c in CH_IN); OUT = sum(TV[c] for c in CH_OUT); NET = IN - OUT
    for k in range(6):
        o[f"net_m{k+1}"] = NET[:, k]; o[f"io_ratio_m{k+1}"] = OUT[:, k] / (IN[:, k] + E)
    in_mean, out_mean = IN.mean(1), OUT.mean(1)
    o["in_mean"] = in_mean; o["out_mean"] = out_mean; o["net_sum"] = NET.sum(1); o["net_m13"] = NET[:, :3].sum(1)
    o["io_ratio"] = OUT.sum(1) / (IN.sum(1) + E); o["io_ratio_m13"] = OUT[:, :3].sum(1) / (IN[:, :3].sum(1) + E)
    o["in_slope"] = slope(IN) / (in_mean + E); o["out_slope"] = slope(OUT) / (out_mean + E); o["net_slope"] = slope(NET)
    o["in_cv"] = IN.std(1) / (in_mean + E); o["out_cv"] = OUT.std(1) / (out_mean + E)
    o["coverage_m1"] = B[:, 0] / (OUT[:, 0] + E); o["coverage"] = bal_mean / (out_mean + E); o["bal_to_in"] = bal_mean / (in_mean + E)
    o["bal_m1_to_out_mean"] = B[:, 0] / (out_mean + E)
    o["withdraw_share_in"] = TV["withdraw"].sum(1) / (IN.sum(1) + E)
    o["withdraw_share_out"] = TV["withdraw"].sum(1) / (OUT.sum(1) + E)
    o["send_share_out"] = TV["mm_send"].sum(1) / (OUT.sum(1) + E)
    o["bill_share_out"] = (TV["paybill"] + TV["merchantpay"]).sum(1) / (OUT.sum(1) + E)
    o["recv_share_in"] = TV["received"].sum(1) / (IN.sum(1) + E); o["bank_share_in"] = TV["transfer_from_bank"].sum(1) / (IN.sum(1) + E)
    o["dep_share_in"] = TV["deposit"].sum(1) / (IN.sum(1) + E)
    VT = sum(V[c] for c in CH)
    o["vol_total"] = VT.sum(1); o["vol_slope"] = slope(VT) / (VT.mean(1) + E); o["vol_m1_mean"] = VT[:, 0] / (VT.mean(1) + E)
    arpu = df["arpu"].values
    o["arpu_to_bal"] = arpu / (bal_mean + E); o["arpu_to_in"] = arpu / (in_mean + E); o["arpu_to_out"] = arpu / (out_mean + E)
    R = np.column_stack([(B[:, k] - B[:, k + 1]) - NET[:, k] for k in range(5)])   # balance change not explained by flows
    for k in range(5):
        o[f"resid_m{k+1}"] = R[:, k]
    o["resid_abs_mean"] = np.abs(R).mean(1)

    # Window contrasts: recent s months vs the remaining months, for every series
    S = {"bal": B, "in": IN, "out": OUT, "vt": VT}
    for c in CH:
        S[c + "_tv"] = TV[c]; S[c + "_v"] = V[c]; S[c + "_u"] = mat(df, "m{}_" + c + "_" + UNIQ[c])
    for key, A in S.items():
        for s in (1, 2, 3):
            r, p = A[:, :s].mean(1), A[:, s:].mean(1)
            o[f"w{s}_{key}_pct"] = (r - p) / (np.abs(p) + E)
            if s != 3:
                o[f"w{s}_{key}_ratio"] = r / (p + E)
            o[f"w{s}_{key}_diff"] = r - p
    for s in (1, 2, 3):
        o[f"w{s}_net_diff"] = NET[:, :s].mean(1) - NET[:, s:].mean(1)
        o[f"w{s}_io_diff"] = OUT[:, :s].sum(1) / (IN[:, :s].sum(1) + E) - OUT[:, s:].sum(1) / (IN[:, s:].sum(1) + E)
    o["io_ratio_m46"] = OUT[:, 3:].sum(1) / (IN[:, 3:].sum(1) + E)
    o["net_m46"] = NET[:, 3:].sum(1)
    o["bal_m13_minus_m46_over_out"] = (B[:, :3].mean(1) - B[:, 3:].mean(1)) / (out_mean + E)
    return pd.DataFrame(o, index=df.index)

### Stress patterns, missed income and profile-relative features
Group medians come from the training rows only and never use the target.

In [ ]:
# New features: stress-pattern counts, missed-income flags, profile-relative scores.

def _run_len_from_start(M):
    """Length of the run of True values starting at column 0 (M1 backwards in time)."""
    out = np.zeros(len(M), int); alive = np.ones(len(M), bool)
    for k in range(M.shape[1]):
        alive &= M[:, k]; out += alive
    return out

def _longest_run(M):
    best = np.zeros(len(M), int); cur = np.zeros(len(M), int)
    for k in range(M.shape[1]):
        cur = np.where(M[:, k], cur + 1, 0); best = np.maximum(best, cur)
    return best

def build_new(df, ref):
    """df: raw rows to featurise; ref: raw TRAIN rows used for group medians (target not used)."""
    o = {}
    B = mat(df, "m{}_daily_avg_bal")
    TV = {c: mat(df, "m{}_" + c + "_total_value") for c in CH}
    V = {c: mat(df, "m{}_" + c + "_volume") for c in CH}
    IN = sum(TV[c] for c in CH_IN); OUT = sum(TV[c] for c in CH_OUT); NET = IN - OUT
    # --- stress-pattern counts
    o["n_out_gt_in"] = (NET < 0).sum(1)
    o["streak_out_gt_in"] = _run_len_from_start(NET < 0)          # consecutive deficit months ending at M1
    dec = B[:, :-1] < B[:, 1:]                                    # balance fell vs the month before (m_k < m_k+1)
    o["streak_bal_decline"] = _run_len_from_start(dec)
    o["longest_bal_decline"] = _longest_run(dec[:, ::-1])
    o["bal_m1_is_min"] = (B[:, 0] <= B.min(1)).astype(int)
    o["bal_m1_below_prev_min"] = (B[:, 0] < B[:, 1:].min(1)).astype(int)
    o["bal_m1_rank"] = (B[:, [0]] > B[:, 1:]).sum(1)
    o["bal_drawdown"] = (B.max(1) - B[:, 0]) / (B.max(1) + E)
    o["bal_m1_cover_prev_out"] = B[:, 0] / (OUT[:, 1:].mean(1) + E)
    cash_out = TV["withdraw"]
    o["cashout_share_m1"] = cash_out[:, 0] / (IN[:, 0] + E)
    o["cashout_share_mean"] = (cash_out / (IN + E)).mean(1)
    o["cashout_share_m1_vs_mean"] = o["cashout_share_m1"] - o["cashout_share_mean"]
    o["n_channels_m1"] = sum((V[c][:, 0] > 0).astype(int) for c in CH)
    o["n_channels_mean"] = sum((V[c] > 0).mean(1) for c in CH)
    o["channels_drop"] = o["n_channels_mean"] - o["n_channels_m1"]
    VT = sum(V[c] for c in CH)
    o["vol_m1_vs_m23"] = VT[:, 0] / (VT[:, 1:3].mean(1) + E)
    o["act_x_vol_drop"] = df["x_90_d_activity_rate"].values * o["vol_m1_vs_m23"]
    # --- missed expected income
    in_med = np.median(IN[:, 1:], 1)
    o["in_m1_vs_med"] = IN[:, 0] / (in_med + E)
    o["in_active_months"] = (IN > 0).sum(1)
    o["in_m1_zero"] = (IN[:, 0] == 0).astype(int)
    o["in_miss_m1"] = ((IN[:, 0] < 0.5 * in_med) & (in_med > 0)).astype(int)
    bank_act = (V["transfer_from_bank"][:, 1:] > 0).sum(1)
    o["bank_miss_m1"] = ((V["transfer_from_bank"][:, 0] == 0) & (bank_act >= 3)).astype(int)
    o["bank_m1_vs_med"] = TV["transfer_from_bank"][:, 0] / (np.median(TV["transfer_from_bank"][:, 1:], 1) + E)
    recv_act = (V["received"][:, 1:] > 0).sum(1)
    o["recv_miss_m1"] = ((V["received"][:, 0] == 0) & (recv_act >= 3)).astype(int)
    ep = df["earning_pattern"].values
    NA = -2.0   # "other earning pattern" code; real values are always > -1 (NaN slows XGBoost evaluation ~3x)
    for lab, tag in [("Monthly Earner", "mon"), ("Weekly Earner", "wk"), ("Irregular Earner", "irr")]:
        is_ = ep == lab
        o[f"in_m1_vs_med_{tag}"] = np.where(is_, o["in_m1_vs_med"], NA)
        o[f"in_miss_m1_{tag}"] = (is_ & (o["in_miss_m1"] == 1)).astype(int)
        o[f"w3_in_pct_{tag}"] = np.where(is_, (IN[:, :3].mean(1) - IN[:, 3:].mean(1)) / (IN[:, 3:].mean(1) + E), NA)
    # --- profile-relative scores (value / group median, groups from TRAIN rows)
    def keyvals(d):
        Bd = mat(d, "m{}_daily_avg_bal")
        INd = sum(mat(d, "m{}_" + c + "_total_value") for c in CH_IN)
        OUTd = sum(mat(d, "m{}_" + c + "_total_value") for c in CH_OUT)
        return pd.DataFrame({"bal_mean": Bd.mean(1), "bal_m1": Bd[:, 0], "in_mean": INd.mean(1), "in_m1": INd[:, 0],
                             "out_mean": OUTd.mean(1), "out_m1": OUTd[:, 0], "arpu": d["arpu"].values,
                             "io_m13": OUTd[:, :3].sum(1) / (INd[:, :3].sum(1) + E)}, index=d.index)
    Kd, Kr = keyvals(df), keyvals(ref)
    for grp in [["segment"], ["earning_pattern"], ["segment", "earning_pattern"], ["region"]]:
        g = "_".join(x[:3] for x in grp)
        kd = df[grp].astype(str).agg("|".join, axis=1).values
        kr = ref[grp].astype(str).agg("|".join, axis=1).values
        med = Kr.groupby(kr).median()
        for col in Kd.columns:
            m = pd.Series(kd).map(med[col]).values
            o[f"rel_{g}_{col}"] = Kd[col].values / (np.abs(m) + E)
    return pd.DataFrame(o, index=df.index)


RAW_TRAIN = train.drop(columns=[ID_COL, TARGET])


def make_matrix(df):
    X = df.drop(columns=[ID_COL, TARGET], errors="ignore").copy()
    out = pd.concat([X, build_features(X), build_new(X, RAW_TRAIN)], axis=1)
    for c in CATS:
        out[c] = pd.Categorical(out[c], categories=sorted(train[c].unique()))
    return out


t0 = time.time()
X_all, X_test_all = make_matrix(train), make_matrix(test)
print("feature matrix:", X_all.shape, X_test_all.shape, f"({time.time()-t0:.1f}s)")

### v5 features: runway and projections
How long the current balance lasts at the recent burn rate, and where the balance lands next month if recent flows repeat.
Trees can only approximate these ratios with many splits, so stating them directly helps.

In [ ]:
def build_v5(df):
    o = {}
    B = mat(df, "m{}_daily_avg_bal")
    TV = {c: mat(df, "m{}_" + c + "_total_value") for c in CH}
    H = {c: mat(df, "m{}_" + c + "_highest_amount") for c in CH}
    IN = sum(TV[c] for c in CH_IN); OUT = sum(TV[c] for c in CH_OUT); NET = IN - OUT
    b1, out_mean = B[:, 0], OUT.mean(1)
    for s in (1, 3, 6):
        deficit = np.maximum(-NET[:, :s].mean(1), 0)
        o[f"runway_{s}"] = b1 / (deficit + E)                                   # months the M1 balance covers the deficit
        o[f"proj_bal_{s}"] = b1 + NET[:, :s].mean(1)                            # balance next month if flows repeat
        o[f"proj_bal_{s}_rel"] = o[f"proj_bal_{s}"] / (out_mean + E)
        o[f"proj_bal_{s}_neg"] = (o[f"proj_bal_{s}"] < 0).astype(int)
    o["proj_bal_trend"] = b1 + slope(B)
    o["proj_bal_trend_rel"] = o["proj_bal_trend"] / (B.mean(1) + E)
    o["bal_m1_to_out_m1"] = b1 / (OUT[:, 0] + E)
    o["bal_m1_to_in_m1"] = b1 / (IN[:, 0] + E)
    o["net_std_rel"] = NET.std(1) / (out_mean + E)
    o["net_worst"] = NET.min(1) / (out_mean + E)
    o["bal_low_months"] = (B < 0.5 * B.mean(1, keepdims=True)).sum(1)
    o["bal_m1_minus_min_rel"] = (b1 - B[:, 1:].min(1)) / (out_mean + E)
    hi_out_m1 = np.max(np.column_stack([H[c][:, 0] for c in CH_OUT]), 1)
    hi_out = np.max(np.column_stack([H[c].max(1) for c in CH_OUT]), 1)
    o["hi_out_m1_to_bal"] = hi_out_m1 / (b1 + E)
    o["hi_out_to_bal"] = hi_out / (b1 + E)
    o["hi_out_m1_vs_hist"] = hi_out_m1 / (np.max(np.column_stack([H[c][:, 1:].max(1) for c in CH_OUT]), 1) + E)
    o["out_m1_to_in_mean"] = OUT[:, 0] / (IN.mean(1) + E)
    o["in_m1_to_out_mean"] = IN[:, 0] / (out_mean + E)
    o["arpu_to_bal_m1"] = df["arpu"].values / (b1 + E)
    return pd.DataFrame(o, index=df.index)


V5_train, V5_test = build_v5(train.drop(columns=[ID_COL, TARGET])), build_v5(test.drop(columns=[ID_COL]))
print("v5 features:", V5_train.shape[1])

## Feature selection: top 250 by LightGBM gain (identical to v1-v3, so cached predictions stay valid)

In [ ]:
skf_fs = list(StratifiedKFold(10, shuffle=True, random_state=SEED).split(X_all, y))
fs_params = dict(n_estimators=4000, learning_rate=0.04, num_leaves=31, min_child_samples=50, subsample=0.8,
                 subsample_freq=1, colsample_bytree=0.3, reg_lambda=2, verbose=-1, random_state=SEED, n_jobs=THREADS)
gain, oof_fs = pd.Series(0.0, index=X_all.columns), np.zeros(len(y))
for tr_idx, va_idx in skf_fs:
    m = lgb.LGBMClassifier(**fs_params)
    m.fit(X_all.iloc[tr_idx], y[tr_idx], eval_set=[(X_all.iloc[va_idx], y[va_idx])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    oof_fs[va_idx] = m.predict_proba(X_all.iloc[va_idx])[:, 1]
    gain += pd.Series(m.booster_.feature_importance("gain"), index=X_all.columns)
print("first-pass LGB (all features)  LL %.5f  AUC %.5f" % (log_loss(y, oof_fs), roc_auc_score(y, oof_fs)))
RANKED = list(gain.sort_values(ascending=False).index)
FEATURES = RANKED[:250]
X, X_test = X_all[FEATURES], X_test_all[FEATURES]
cat_in = [c for c in FEATURES if c in CATS]
print("selected:", len(FEATURES), "| categoricals kept:", cat_in)

# plus view: the 250 selected features + the v5 runway/projection features
X_plus, X_test_plus = pd.concat([X, V5_train], axis=1), pd.concat([X_test, V5_test], axis=1)
VIEWS = {"main": (X, X_test, cat_in), "plus": (X_plus, X_test_plus, cat_in)}
for k, (a, b, c) in VIEWS.items():
    print(f"view {k:5s}: {a.shape[1]} features")

### Quick check: do the v5 features help? (one LightGBM seed, 5 folds, about a minute)

In [ ]:
def quick_lgb(Xm, sd=42):
    oof = np.zeros(len(y))
    for tr_idx, va_idx in StratifiedKFold(5, shuffle=True, random_state=sd).split(Xm, y):
        m = lgb.LGBMClassifier(n_estimators=6000, learning_rate=0.03, num_leaves=7, min_child_samples=100,
                               colsample_bytree=0.3, reg_lambda=5, subsample=0.8, subsample_freq=1,
                               verbose=-1, random_state=sd, n_jobs=THREADS)
        m.fit(Xm.iloc[tr_idx], y[tr_idx], eval_set=[(Xm.iloc[va_idx], y[va_idx])], callbacks=[lgb.early_stopping(200, verbose=False)])
        oof[va_idx] = m.predict_proba(Xm.iloc[va_idx])[:, 1]
    return log_loss(y, oof), roc_auc_score(y, oof)

for k in VIEWS:
    print(f"{k:5s} view  LL %.5f  AUC %.5f" % quick_lgb(VIEWS[k][0]))

## Models (stratified 10-fold CV; results cached in preds/)
lgb, cb, cb_ord load from the v2/v3 cache. New models average their fold test predictions in logit space
(cache files tagged `_lt`), and every model also saves its mean best iteration.

In [ ]:
def run_cv(name, fit_predict, seeds, view="main", logit_avg=False):
    Xm, Xtm, cats = VIEWS[view]
    oof_s, pt_s = [], []
    for sd in seeds:
        key = f"{CACHE_DIR}/{name}_f{N_FOLDS}_s{sd}_fs{SEED}" + ("_lt" if logit_avg else "")
        if os.path.exists(key + "_oof.npy"):
            oof, pt = np.load(key + "_oof.npy"), np.load(key + "_pt.npy")
            print(f"  {name} seed {sd}: loaded from cache", end="")
        else:
            oof, acc, iters = np.zeros(len(y)), np.zeros(len(Xtm)), []
            for tr_idx, va_idx in StratifiedKFold(N_FOLDS, shuffle=True, random_state=sd).split(Xm, y):
                p_va, p_te, it = fit_predict(Xm.iloc[tr_idx], y[tr_idx], Xm.iloc[va_idx], y[va_idx], Xtm, sd, cats)
                oof[va_idx] = p_va; iters.append(it)
                acc += (logit(np.clip(p_te, 1e-7, 1 - 1e-7)) if logit_avg else p_te) / N_FOLDS
            pt = expit(acc) if logit_avg else acc
            np.save(key + "_oof.npy", oof); np.save(key + "_pt.npy", pt)
            print(f"  {name} seed {sd} (mean best iter {np.mean(iters):.0f}):", end="")
        print(f" LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}", flush=True)
        oof_s.append(oof); pt_s.append(pt)
    oof, pt = np.mean(oof_s, 0), np.mean(pt_s, 0)
    print(f"{name} seed-avg: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}")
    return oof, pt


def fp_lgb(Xa, ya, Xb, yb, Xt, sd, cats):
    m = lgb.LGBMClassifier(n_estimators=6000, learning_rate=0.02, num_leaves=7, min_child_samples=100,
                           colsample_bytree=0.3, reg_lambda=5, subsample=0.8, subsample_freq=1,
                           verbose=-1, random_state=sd, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], callbacks=[lgb.early_stopping(300, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1], m.best_iteration_


def fp_lgb_plus(Xa, ya, Xb, yb, Xt, sd, cats):
    # extra-trees split finding + path smoothing: different errors from fp_lgb, similar strength
    m = lgb.LGBMClassifier(n_estimators=10000, learning_rate=0.02, num_leaves=15, min_child_samples=150,
                           colsample_bytree=0.25, reg_lambda=10, subsample=0.8, subsample_freq=1,
                           extra_trees=True, path_smooth=10, verbose=-1, random_state=sd, n_jobs=THREADS)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], callbacks=[lgb.early_stopping(400, verbose=False)])
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1], m.best_iteration_


def make_fp_cb(ordered=False, depth=5, lr=0.04, l2=5, rsm=0.3):
    def fp(Xa, ya, Xb, yb, Xt, sd, cats):
        extra = {"boosting_type": "Ordered"} if ordered else {}
        m = CatBoostClassifier(iterations=6000, learning_rate=lr, depth=depth, l2_leaf_reg=l2, rsm=rsm,
                               loss_function="Logloss", random_seed=sd, od_type="Iter", od_wait=300,
                               verbose=0, thread_count=THREADS, cat_features=cats, **extra)
        m.fit(Xa, ya, eval_set=(Xb, yb), use_best_model=True)
        return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1], m.get_best_iteration()
    return fp


def fp_xgb(Xa, ya, Xb, yb, Xt, sd, cats):
    m = xgb.XGBClassifier(n_estimators=6000, learning_rate=0.02, max_depth=4, min_child_weight=5, subsample=0.8,
                          colsample_bytree=0.3, reg_lambda=5, tree_method="hist", enable_categorical=True,
                          max_cat_to_onehot=8, eval_metric="logloss", early_stopping_rounds=300,
                          random_state=sd, n_jobs=THREADS, verbosity=0)
    m.fit(Xa, ya, eval_set=[(Xb, yb)], verbose=False)
    return m.predict_proba(Xb)[:, 1], m.predict_proba(Xt)[:, 1], m.best_iteration


OOFS, PTS = {}, {}
def timed(name, *args, **kw):
    t0 = time.time(); OOFS[name], PTS[name] = run_cv(name, *args, **kw); print(f"({(time.time()-t0)/60:.1f} min)\n")

timed("lgb", fp_lgb, SEEDS_LGB)
timed("cb", make_fp_cb(), SEEDS_CB)
timed("cb_ord", make_fp_cb(ordered=True), SEEDS_CB_ORD)
timed("xgb", fp_xgb, SEEDS_XGB, logit_avg=True)
timed("lgb_plus", fp_lgb_plus, SEEDS_LGB_PLUS, view="plus", logit_avg=True)
timed("cb_plus", make_fp_cb(depth=6, lr=0.03, l2=8, rsm=0.25), SEEDS_CB_PLUS, view="plus", logit_avg=True)

## Neural network (loaded from the v2/v3 cache; retrained with Keras only if the cache is missing)

In [ ]:
NN_NUM = [c for c in RANKED if c not in CATS][:45]
D = pd.get_dummies(pd.concat([X_all[CATS], X_test_all[CATS]]).astype(str), dtype=float).values
D_tr, D_te = D[:len(X_all)], D[len(X_all):]
print("NN inputs:", len(NN_NUM), "numeric +", D.shape[1], "one-hot")


def build_nn(n_in):
    import keras
    from keras import layers
    reg = keras.regularizers.l2(1e-5)
    inp = keras.Input((n_in,))
    h = layers.Dense(128, activation="relu", kernel_regularizer=reg)(inp)
    h = layers.BatchNormalization()(h)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(64, activation="relu", kernel_regularizer=reg)(h)
    h = layers.Dropout(0.3)(h)
    out = layers.Dense(2, activation="softmax")(h)
    m = keras.Model(inp, out)
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy")
    return m


def run_nn(seeds):
    oof_s, pt_s = [], []
    for sd in seeds:
        key = f"{CACHE_DIR}/nn_f{N_FOLDS}_s{sd}_fs{SEED}"
        if os.path.exists(key + "_oof.npy"):
            oof, pt = np.load(key + "_oof.npy"), np.load(key + "_pt.npy")
        else:
            oof, pt = np.zeros(len(y)), np.zeros(len(X_test_all))
            for tr_idx, va_idx in StratifiedKFold(N_FOLDS, shuffle=True, random_state=sd).split(X_all, y):
                qt = QuantileTransformer(n_quantiles=1000, output_distribution="normal", random_state=sd)
                Xa = np.hstack([np.nan_to_num(qt.fit_transform(X_all[NN_NUM].iloc[tr_idx])), D_tr[tr_idx]])
                Xb = np.hstack([np.nan_to_num(qt.transform(X_all[NN_NUM].iloc[va_idx])), D_tr[va_idx]])
                Xt = np.hstack([np.nan_to_num(qt.transform(X_test_all[NN_NUM])), D_te])
                import keras
                keras.utils.set_random_seed(sd)
                m = build_nn(Xa.shape[1])
                m.fit(Xa, y[tr_idx], validation_data=(Xb, y[va_idx]), epochs=200, batch_size=1024, verbose=0,
                      callbacks=[keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True),
                                 keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4, min_lr=1e-5)])
                oof[va_idx] = m.predict(Xb, verbose=0, batch_size=4096)[:, 1]
                pt += m.predict(Xt, verbose=0, batch_size=4096)[:, 1] / N_FOLDS
            np.save(key + "_oof.npy", oof); np.save(key + "_pt.npy", pt)
        print(f"  NN seed {sd}: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}", flush=True)
        oof_s.append(oof); pt_s.append(pt)
    oof, pt = np.mean(oof_s, 0), np.mean(pt_s, 0)
    print(f"NN seed-avg: LL {log_loss(y, oof):.5f}  AUC {roc_auc_score(y, oof):.5f}")
    return oof, pt


t0 = time.time(); OOFS["nn"], PTS["nn"] = run_nn(SEEDS_NN); print(f"({(time.time()-t0)/60:.1f} min)")

## Blend: compare blending methods with weights fitted out of sample, keep the best
Every candidate is scored the same way: weights fitted on 4/5 of the out-of-fold rows, scored on the other 1/5 (rotated).
- `constrained`: non-negative weights summing to 1 in logit space (v2 method), all models
- `greedy`: forward selection, adding the model that most improves the nested score, stop when nothing helps
- `stack`: free weights + intercept in logit space (a logistic regression on model logits, light L2 toward the constrained solution)

In [ ]:
def ll_fast(yy, p):
    p = np.clip(p, 1e-15, 1 - 1e-15); return float(-np.mean(yy * np.log(p) + (1 - yy) * np.log1p(-p)))

def fit_weights(Lm, yy):
    """Non-negative weights summing to 1 (constrained SLSQP)."""
    k = Lm.shape[1]
    if k == 1: return np.ones(1)
    r = minimize(lambda w: ll_fast(yy, expit(Lm @ w)), np.ones(k) / k, method="SLSQP", bounds=[(0, 1)] * k,
                 constraints=[{"type": "eq", "fun": lambda w: w.sum() - 1}], options={"maxiter": 1000, "ftol": 1e-10})
    w = np.clip(r.x, 0, None)
    return w / w.sum()

def fit_stack(Lm, yy, l2=1e-3):
    """Free weights + intercept; L2 pulls toward the constrained weights so small folds stay stable."""
    w0 = fit_weights(Lm, yy)
    obj = lambda t: ll_fast(yy, expit(t[0] + Lm @ t[1:])) + l2 * np.sum((t[1:] - w0) ** 2)
    return minimize(obj, np.r_[0.0, w0], method="L-BFGS-B").x

L = {k: logit(np.clip(v, 1e-6, 1 - 1e-6)) for k, v in OOFS.items()}
LT = {k: logit(np.clip(v, 1e-6, 1 - 1e-6)) for k, v in PTS.items()}
OUTER = list(StratifiedKFold(5, shuffle=True, random_state=123).split(L["lgb"], y))

def nested(cols, method="constrained"):
    Lm = np.column_stack([L[c] for c in cols]); pred = np.zeros(len(y))
    for a, b in OUTER:
        if method == "stack":
            t = fit_stack(Lm[a], y[a]); pred[b] = expit(t[0] + Lm[b] @ t[1:])
        else:
            pred[b] = expit(Lm[b] @ fit_weights(Lm[a], y[a]))
    return ll_fast(y, pred), roc_auc_score(y, pred)

print("OOF correlation:\n", pd.DataFrame(np.corrcoef(np.column_stack(list(OOFS.values())).T), index=OOFS, columns=OOFS).round(4))
ALL = list(OOFS)
V2_SET = ["lgb", "cb", "cb_ord", "nn"]

# greedy forward selection on the nested score
chosen, best = [], (np.inf, 0)
while True:
    trials = [(nested(chosen + [c]), c) for c in ALL if c not in chosen]
    if not trials: break
    (llv, aucv), c = min(trials, key=lambda t: t[0][0])
    if llv >= best[0] - 1e-6: break
    chosen.append(c); best = (llv, aucv)
    print(f"  greedy + {c:9s} nested LL {llv:.5f}  AUC {aucv:.5f}")

CANDS = {"v2 set, constrained": (V2_SET, "constrained"), "all, constrained": (ALL, "constrained"),
         "greedy, constrained": (chosen, "constrained"), "greedy, stack": (chosen, "stack"), "all, stack": (ALL, "stack")}
rows = []
for nm, (cols, meth) in CANDS.items():
    llv, aucv = nested(cols, meth); rows.append((llv, aucv, nm, cols, meth))
    print(f"{nm:22s} nested LL {llv:.5f}  AUC {aucv:.5f}   {cols}")
best_ll, best_auc, best_name, best_cols, best_meth = min(rows)
print(f"\nkept: {best_name}   (v2 reference: nested LL 0.23055 / AUC 0.92030)")

Lb, Lbt = np.column_stack([L[c] for c in best_cols]), np.column_stack([LT[c] for c in best_cols])
if best_meth == "stack":
    t = fit_stack(Lb, y); oof_blend, test_pred = expit(t[0] + Lb @ t[1:]), expit(t[0] + Lbt @ t[1:])
    print("intercept %.3f | weights:" % t[0], dict(zip(best_cols, t[1:].round(3))))
else:
    W = fit_weights(Lb, y); oof_blend, test_pred = expit(Lb @ W), expit(Lbt @ W)
    print("weights:", dict(zip(best_cols, W.round(3))))
print("Blend OOF (in-sample weights)  LL %.5f  AUC %.5f" % (log_loss(y, oof_blend), roc_auc_score(y, oof_blend)))
print("mean pred: OOF %.4f | test %.4f | train rate %.4f" % (oof_blend.mean(), test_pred.mean(), y.mean()))

## Optional: pseudo-labelling (RUN_PSEUDO)
The test set is 75% the size of train. Confident test rows (blend p < 0.3% or > 97%) are added to every training fold with hard
labels, and a LightGBM is retrained on the `plus` view. Out-of-fold scores stay on original rows only, but the pseudo labels came
from models that saw those rows, so the CV gain here is optimistic. Judge this one on the leaderboard.

In [ ]:
if RUN_PSEUDO:
    LO, HI = 0.003, 0.97
    pl = np.where((test_pred < LO) | (test_pred > HI))[0]
    X_pl, y_pl = X_test_plus.iloc[pl], (test_pred[pl] > 0.5).astype(int)
    print(f"pseudo rows: {len(pl)} ({y_pl.mean():.3f} positive)")

    def fp_lgb_pl(Xa, ya, Xb, yb, Xt, sd, cats):
        return fp_lgb_plus(pd.concat([Xa, X_pl]), np.r_[ya, y_pl], Xb, yb, Xt, sd, cats)

    timed("lgb_pl", fp_lgb_pl, SEEDS_LGB_PLUS, view="plus", logit_avg=True)
    L["lgb_pl"] = logit(np.clip(OOFS["lgb_pl"], 1e-6, 1 - 1e-6)); LT["lgb_pl"] = logit(np.clip(PTS["lgb_pl"], 1e-6, 1 - 1e-6))
    cols = best_cols + ["lgb_pl"]
    llv, aucv = nested(cols, best_meth)
    print(f"{best_name} + lgb_pl  nested LL {llv:.5f}  AUC {aucv:.5f}  (optimistic, see above)")
    Lp, Lpt = np.column_stack([L[c] for c in cols]), np.column_stack([LT[c] for c in cols])
    if best_meth == "stack":
        t = fit_stack(Lp, y); test_pred_pl = expit(t[0] + Lpt @ t[1:])
    else:
        test_pred_pl = expit(Lpt @ fit_weights(Lp, y))
    pd.DataFrame({"ID": test[ID_COL], "Target": test_pred_pl}).to_csv("submissionv5_pseudo.csv", index=False)
    print("wrote submissionv5_pseudo.csv")

## Submissions (raw probabilities)

In [ ]:
submission = pd.DataFrame({"ID": test[ID_COL], "Target": test_pred})
assert submission.shape[0] == test.shape[0] and submission.Target.between(0, 1).all() and submission.ID.is_unique
submission.to_csv("submissionv5.csv", index=False)
print("submissionv5.csv", submission.shape, "| mean pred %.4f (train rate %.4f)" % (test_pred.mean(), y.mean()))

# Free extra candidate: logit average of the two best leaderboard files
if os.path.exists("submissionv1.csv") and os.path.exists("submissionv2.csv"):
    s1 = pd.read_csv("submissionv1.csv")
    s2 = pd.read_csv("submissionv2.csv").set_index("ID").loc[s1.ID].reset_index()
    p12 = expit(0.5 * logit(np.clip(s1.Target, 1e-6, 1 - 1e-6)) + 0.5 * logit(np.clip(s2.Target, 1e-6, 1 - 1e-6)))
    pd.DataFrame({"ID": s1.ID, "Target": p12}).to_csv("submission_v1v2_logitavg.csv", index=False)
    print("submission_v1v2_logitavg.csv | corr v1 vs v2: %.4f" % np.corrcoef(s1.Target, s2.Target)[0, 1])
submission.head()